이 노트북을 실행하는 데 필요한 라이브러리(표준 라이브러리 제외)
- torch
- numpy
- matplotlib

예제 실행 및 시각화를 위해 공통 라이브러리를 불러온다.
(공통 라이브러리에 대한 설명은 `code_reference/README.md` 파일을 참조)


In [1]:
# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz
# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 6-1 순환 신경망

본 노트북은 본문 6-1절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 순차 데이터를 다루기 위한 어휘 사전과 원-핫 인코딩
- 슬라이딩 윈도우로 같은 길이의 데이터 샘플 재구성
- 특수 토큰(`<eos>`, `<pad>`)의 쓰임새

순서가 의미를 만드는 순차 데이터를 신경망에 넣을 수 있는 형태로 바꾸는 과정을 다룬다.

## 어휘 사전과 원-핫 인코딩

- 순차 데이터를 구성하는 각 단위를 **토큰**이라 하고, 토큰과 고유 번호를 짝지은 표를 **어휘 사전**이라 한다.
- 토큰의 고유 번호는 크기에 의미가 없으므로, 번호의 대소가 학습에 영향을 주지 않도록 원-핫 인코딩해 사용한다.

In [2]:
######################################################################################
# 코드 6-1 - 어휘 사전과 원-핫 인코딩
######################################################################################

import torch
import torch.nn.functional as F

# 어휘 사전: 보통 고유 번호는 0부터 차례대로 부여
#            토큰에 따른 번호의 순서는 의미가 없으므로 가나다순으로 바꿔 매겨도 무방
#            어휘 사전의 변수명은 보통 vocab을 사용
vocab = {'도': 0, '레': 1, '미': 2, '파': 3, '솔': 4, '라': 5, '시': 6}

# '도도솔솔'의 원-핫 인코딩 과정(3장의 F.one-hot() 사용)
sequence = '도도솔솔'
# 음을 어휘 사전의 고유 번호 텐서로 변환
sequence_idx = torch.tensor([vocab[token] for token in sequence])
print('고유 번호의 텐서:', sequence_idx)
# 어휘 사전의 크기 = 고유한 요소의 개수
vocab_size = len(vocab)                
# 원-핫 인코딩 후 모델의 학습 데이터로 사용하려면 실수형 텐서로 변환해야 함(float())
sequence_onehot = F.one_hot(sequence_idx, num_classes=vocab_size).float()

print('원-핫 인코딩된 텐서:')
print(sequence_onehot)

고유 번호의 텐서: tensor([0, 0, 4, 4])
원-핫 인코딩된 텐서:
tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.]])


## 슬라이딩 윈도우

- 순환 신경망은 같은 길이의 샘플을 묶어 배치로 학습한다.
    - 길이가 다른 순차 데이터를 일정한 크기의 창으로 훑어 같은 길이의 샘플을 여러 개 만든다.
- 창을 한 칸씩 옮기며 잘라내므로, 하나의 순차 데이터에서 여러 샘플이 나온다.

In [3]:
######################################################################################
# 코드 6-2 - '도도솔솔라라솔'을 같은 길이의 데이터 샘플로 재구성
######################################################################################

sequence = '도도솔솔라라솔'
# 이전 음 4개로 다음 음을 예측하기 위해 크기 5인 윈도우를 사용(입력 4개 + 정답 1개)
window_size = 5                                        # 윈도우의 크기
dataset = []
for i in range(len(sequence) - window_size + 1):
    subsequence = sequence[i: i + window_size]
    # 입력: 마지막을 뺀 앞 요소들, 정답: 마지막 1개 요소
    dataset.append((subsequence[:-1], subsequence[-1]))

print('입력 데이터 -> 정답')
for input_data, label in dataset:
    print(f'{input_data}    -> {label}')

입력 데이터 -> 정답
도도솔솔    -> 라
도솔솔라    -> 라
솔솔라라    -> 솔


## 특수 토큰

- 실제 데이터에는 길이가 제각각인 순차 데이터가 섞여 있어 특수 토큰이 필요하다.
    - `<eos>`: 순차 데이터의 끝을 표시해 어디서 멈춰야 하는지 알려준다.
    - `<pad>`: 길이를 맞추기 위해 빈자리를 채운다.
- 특수 토큰도 어휘 사전에 포함해 고유 번호를 부여한다.

In [4]:
######################################################################################
# 코드 6-3 - <eos>와 <pad>를 추가한 데이터 샘플
######################################################################################

samples = []
# 문자열을 리스트로 바꾼 후, 순차 데이터의 끝에 <eos> 추가
# 토큰 하나를 추가하는 것이지, <eos>라는 다섯 글자의 문자열을 추가하는 것이 아님
sequence_list = list(sequence)
sequence_list.append('<eos>')
# 순차 데이터의 끝부분까지 포함하도록 윈도우를 끝까지 밀어 가며 샘플 생성
for i in range(len(sequence_list) - 1):
    subsequence = sequence_list[i: i + window_size]
    # 모든 샘플의 길이가 같아지도록 부족한 만큼 <pad>로 채움
    while len(subsequence) < window_size:
        subsequence.append('<pad>')
    samples.append((subsequence[:4], subsequence[-1]))

# 추후 원-핫 인코딩을 해야 한다면 어휘 사전에 특수 토큰도 추가해야 함
last_index = len(vocab)
vocab['<eos>'] = last_index             # <eos> 토큰 추가
vocab['<pad>'] = last_index + 1         # <pad> 토큰 추가

print('입력 데이터 -> 정답')
for input_data, label in samples:
    print(f'{input_data} -> {label}')

입력 데이터 -> 정답
['도', '도', '솔', '솔'] -> 라
['도', '솔', '솔', '라'] -> 라
['솔', '솔', '라', '라'] -> 솔
['솔', '라', '라', '솔'] -> <eos>
['라', '라', '솔', '<eos>'] -> <pad>
['라', '솔', '<eos>', '<pad>'] -> <pad>
['솔', '<eos>', '<pad>', '<pad>'] -> <pad>


In [5]:
# 참고 - 특수 토큰이 추가된 어휘 사전 확인

print('확장된 어휘 사전:', vocab)
print(f'어휘 사전의 크기: {len(vocab)}')

확장된 어휘 사전: {'도': 0, '레': 1, '미': 2, '파': 3, '솔': 4, '라': 5, '시': 6, '<eos>': 7, '<pad>': 8}
어휘 사전의 크기: 9


- 연습문제 6-1에서 사용할 김소월의 시 데이터다.

In [6]:
######################################################################################
# 코드 6-4 - 김소월의 시, '엄마야 누나야' (연습문제 6-1)
######################################################################################

poem = '엄마야 누나야 강변 살자 ' \
       '뜰에는 반짝이는 금모래 빛 ' \
       '뒷문 밖에는 갈잎의 노래 ' \
       '엄마야 누나야 강변 살자'

print(poem)

엄마야 누나야 강변 살자 뜰에는 반짝이는 금모래 빛 뒷문 밖에는 갈잎의 노래 엄마야 누나야 강변 살자


## 정리

- 순차 데이터는 토큰 단위로 나눠 어휘 사전으로 고유 번호를 부여하고, 번호의 크기가 학습에 영향을 주지 않도록 원-핫 인코딩한다.
- 슬라이딩 윈도우로 잘라내면 길이가 다른 순차 데이터에서 같은 길이의 샘플을 여러 개 만들 수 있다.
- `<eos>`는 순차 데이터의 끝을, `<pad>`는 길이를 맞추기 위한 빈자리를 나타내며 어휘 사전에 함께 넣는다.